# Practical 1

## To Develop a Supervised Machine Learning Regression Model for Predicting the Aqueous Solubility (logS) of Drug Molecules from Physicochemical Descriptors

**Objectives**

- To understand what a machine learning model is and how it learns a relationship from data.
- To understand aqueous solubility, the logS scale, and why solubility governs whether a drug can work at all.
- To convert a table of molecular physicochemical properties into the input (X) and output (y) that a machine learning algorithm requires.
- To split a dataset correctly into a training portion and a testing portion, and to explain why that separation is essential.
- To build, train and interpret a Multiple Linear Regression model on real experimental measurements.
- To evaluate a regression model using MAE, MSE, RMSE and the R-squared score, and to confirm the result with cross-validation.
- To diagnose model quality visually using an actual-versus-predicted plot and a residual plot.
- To recognise multicollinearity in real chemical data and understand how it distorts the interpretation of coefficients.
- To save a trained model and apply it to a drug molecule the model has never seen.

**Practical type:** Regression  |  **Approach:** Machine Learning (ML)  |  **Learning type:** Supervised

**Data:** 1,128 real compounds with experimentally measured aqueous solubility

In [ ]:
# Block 1: Import the Python libraries this practical needs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully.")

**Meaning of Block 1**

**Purpose:** Gather every software tool the practical will use, in one place, before any data is touched.

**What data comes IN:** Nothing. This is the starting point of the notebook.

**What happens inside — line by line:**

A *library* is a collection of ready-written programs that other people have already built and tested. Rather than writing thousands of lines of code yourself, you borrow theirs.

- `pandas` reads Excel and CSV files and holds them as a table. The nickname `pd` is a worldwide convention.
- `numpy` handles numerical arrays and mathematics. Nicknamed `np`.
- `matplotlib.pyplot` draws graphs. Nicknamed `plt`.
- `joblib` saves a trained model to a file so it can be reused without retraining.
- `train_test_split` divides a dataset into a learning portion and an examination portion.
- `cross_val_score` repeats the entire train-and-test cycle several times over different portions of the data, to check that a good result was not simply luck.
- `LinearRegression` is the machine learning algorithm used in this practical.
- `mean_absolute_error`, `mean_squared_error` and `r2_score` are the measuring instruments used to judge the trained model.

**What comes OUT:** No data yet — only a confirmation that the tools are loaded.

**Pharmacy analogy:** Before beginning any experiment a pharmacist assembles the balance, pipette, beaker and pH meter on the bench. This block assembles the software instruments.

In [ ]:
# Block 2: Load the solubility dataset

url = "https://raw.githubusercontent.com/drpharmacy/pharma-ml-projects/main/Excel_Files/delaney_solubility_dataset.xlsx"

df = pd.read_excel(url)

print("Dataset loaded successfully.")
print("Number of compounds:", df.shape[0])
print()
df.head()

**Meaning of Block 2**

**Purpose:** Bring the dataset from the internet into this notebook's memory.

**What data comes IN:** Nothing local. The file is fetched directly from a web address.

**What happens inside — line by line:**

- `url = "..."` stores the complete web address of the data file. It is an address in exactly the sense a postal address is: it names the account, the repository, the folder (`Excel_Files`) and finally the file itself. If any part is misspelled, the file will not be found.
- `pd.read_excel(url)` downloads the file and turns it into a **DataFrame** — pandas' name for a table of rows and columns, essentially an Excel worksheet held in memory.
- `df` is the variable holding that table. A *variable* is simply a labelled box in the computer's memory; from here onward, writing `df` means "the whole dataset".
- `df.head()` shows the first five rows so you can confirm the data arrived intact.

**What comes OUT:** `df` — the complete dataset, used by every block that follows.

**Where this data comes from — the full provenance:**

This is the **Delaney aqueous solubility dataset**, one of the most widely used public benchmarks in pharmaceutical machine learning. It was published by John S. Delaney in 2004 in the *Journal of Chemical Information and Computer Sciences*, in a paper introducing the ESOL method for estimating solubility directly from molecular structure. Delaney assembled the measurements from published experimental sources, and the collection has been used ever since as a standard test bed for solubility prediction. It is distributed openly as part of the DeepChem project.

Every one of the 1,128 entries is a **real chemical compound with a real, laboratory-measured aqueous solubility**. The set is deliberately diverse: it contains marketed drugs such as testosterone, phenytoin, diazepam, warfarin and theophylline, alongside agrochemicals, industrial chemicals and simple organic molecules. That breadth is intentional — a model trained only on drug-like molecules would have little to learn from, because it would never see the extremes.

The six physicochemical descriptors in this file were calculated from each compound's chemical structure (its SMILES string) using RDKit, the standard open-source cheminformatics toolkit. Every SMILES was checked for chemical validity and every duplicate structure removed during preparation, so all 1,128 rows are distinct, valid molecules.

**Pharmacy analogy:** It is like receiving a validated reference compendium of measured physicochemical data, rather than a table someone typed up by hand.

In [ ]:
# Block 3: Explore the dataset before doing anything else

print("Shape of the dataset (rows, columns):", df.shape)
print()

print("Column names:")
print(list(df.columns))
print()

print("Structure and data types:")
df.info()
print()

print("Missing values in each column:")
print(df.isnull().sum())
print()

print("Statistical summary:")
print(df.describe().round(2))

**Meaning of Block 3**

**Purpose:** Understand the dataset thoroughly before building anything on top of it. This stage is called **Exploratory Data Analysis (EDA)**.

**What data comes IN:** `df` from Block 2.

**What happens inside — line by line:**

- `df.shape` returns two numbers, rows and columns. Here it gives **(1128, 9)** — 1,128 compounds, each described by 9 columns.
- `df.columns` lists the column names.
- `df.info()` reports the data type of each column. `object` means text, `float64` a decimal number, `int64` a whole number. It also confirms that all 1,128 entries are present in every column.
- `df.isnull().sum()` counts empty cells per column. Real datasets frequently contain gaps, and most machine learning algorithms simply refuse to run when values are missing. Every count here is zero, so no cleaning is required — but you must always check, never assume.
- `df.describe()` gives count, mean, standard deviation, minimum, maximum and quartiles for every numerical column.

**What comes OUT:** No new variable — only knowledge about the data.

**Understanding the nine columns:**

| Column | Meaning | Role |
|---|---|---|
| `Compound_Name` | The compound's common name | Identifier only — never fed to the model |
| `SMILES` | Its chemical structure written as a text string | Identifier only — never fed to the model |
| `molecular_weight` | Mass of one mole, in g/mol | Input feature |
| `logP` | Logarithm of the octanol/water partition coefficient | Input feature |
| `num_h_donors` | Count of hydrogen bond donor groups (–OH, –NH) | Input feature |
| `num_h_acceptors` | Count of hydrogen bond acceptor atoms (O, N) | Input feature |
| `polar_surface_area` | Area of polar surface, in square Ångströms (TPSA) | Input feature |
| `num_rotatable_bonds` | Count of freely rotating single bonds | Input feature |
| `logS` | **Measured aqueous solubility** | Target — the value to predict |

**What logS means — read this carefully:**

Solubility here is expressed as **logS**, the base-10 logarithm of the compound's solubility measured in moles per litre.

Solubilities span an enormous range, from tens of moles per litre down to less than a millionth of a mole per litre. Handling such numbers directly is impractical, so the logarithm compresses them onto a workable scale.

Because most compounds dissolve at less than 1 mol/L, and the logarithm of any number below 1 is negative, **logS values are usually negative**. The more negative the value, the *less* soluble the compound:

- logS = **+1** means 10 mol/L — extremely soluble
- logS = **−2** means 0.01 mol/L — freely soluble
- logS = **−5** means 0.00001 mol/L — poorly soluble
- logS = **−8** means 0.00000001 mol/L — practically insoluble

**What the summary reveals about this dataset:** logS runs from **+1.58** (acetamide, highly water-soluble) down to **−11.60** (a fully chlorinated biphenyl, essentially insoluble). That is a span of more than thirteen log units — a factor of over ten trillion in actual solubility. The mean sits at about −3.05. Roughly 28 per cent of the compounds are more soluble than logS −1.79, so the set genuinely covers the soluble end of the scale as well as the insoluble.

That range is exactly what a model needs. A dataset containing only poorly soluble compounds could never learn what makes a molecule soluble.

**Why solubility matters in pharmacy:** A drug must dissolve in gastrointestinal fluid before it can be absorbed. A tablet of a poorly soluble drug may pass through the patient with very little of the active ingredient ever reaching the bloodstream. Estimating solubility computationally — before a compound is ever synthesised — saves enormous time and expense in formulation development, and is one of the earliest filters applied in modern drug discovery.

**Pharmacy analogy:** Before starting an experiment, a researcher counts the samples, inspects them, and confirms that no information is missing.

In [ ]:
# Block 4: Check how strongly the variables are related to each other

numeric_columns = ["molecular_weight", "logP", "num_h_donors",
                   "num_h_acceptors", "polar_surface_area",
                   "num_rotatable_bonds", "logS"]

correlation_matrix = df[numeric_columns].corr().round(2)

print("Correlation of each descriptor with logS:")
print(correlation_matrix["logS"].sort_values().to_string())
print()

plt.figure(figsize=(9, 7))
plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation coefficient')
plt.xticks(range(len(numeric_columns)), numeric_columns, rotation=45, ha='right')
plt.yticks(range(len(numeric_columns)), numeric_columns)
plt.title("Correlation between all variables")

for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        plt.text(j, i, correlation_matrix.iloc[i, j],
                 ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Meaning of Block 4**

**Purpose:** Measure how strongly each pair of variables moves together, before fitting a linear model.

**What data comes IN:** `df` from Block 2. Only the numerical columns are used — the name and SMILES columns are text and cannot be correlated.

**What happens inside — line by line:**

- `df[numeric_columns].corr()` calculates the **correlation coefficient** between every pair of columns. This is a number between −1 and +1:
  - **+1** means the two variables rise together perfectly
  - **0** means they are unrelated
  - **−1** means one rises exactly as the other falls
- `plt.imshow(...)` draws the matrix as a coloured grid called a heatmap. Red indicates positive correlation, blue negative, pale colours near zero.
- The nested `for` loops print the actual number inside each square, so the figure can be read precisely rather than by colour alone.

**What comes OUT:** `correlation_matrix` — a 7 × 7 table, plus the heatmap.

**First, read the `logS` row — which descriptors relate to solubility?**

| Descriptor | Correlation with logS | Interpretation |
|---|---|---|
| `logP` | **−0.83** | By far the strongest. Greasier molecules are much less water-soluble |
| `molecular_weight` | **−0.64** | Strong. Larger molecules are harder for water to surround |
| `num_rotatable_bonds` | −0.24 | Weak |
| `num_h_donors` | +0.21 | Weak positive |
| `polar_surface_area` | +0.12 | Very weak |
| `num_h_acceptors` | +0.05 | Essentially none |

The two dominant drivers are lipophilicity and size, exactly as physical pharmaceutics teaches. Seeing that emerge from 1,128 real measurements, with no chemistry knowledge programmed in, is the first genuine result of this practical.

**Second, read the descriptors against each other — the multicollinearity check.**

When two *input* variables are strongly correlated with one another, the condition is called **multicollinearity**, and it makes the coefficients of a linear regression unstable and difficult to interpret. In this real dataset several pairs are strongly related:

- `num_h_acceptors` and `polar_surface_area`: **0.90**
- `num_h_donors` and `polar_surface_area`: **0.76**
- `num_h_donors` and `num_h_acceptors`: **0.58**
- `molecular_weight` and `num_h_acceptors`: **0.56**

The first of these is very high indeed, and it is chemically obvious once stated: polar surface area is calculated largely from the oxygen and nitrogen atoms in a molecule, and those same atoms are what the hydrogen bond acceptor count counts. The two descriptors are measuring nearly the same underlying property in two different ways.

**Why this matters, and why you should welcome seeing it:** When two features carry nearly the same information, the model cannot tell which of them deserves the credit. It may hand a large positive coefficient to one and a compensating negative coefficient to the other, producing an equation that predicts perfectly well while being chemically misleading if read literally. You will see exactly this happen in Block 9, and you will then understand why it happened.

Real chemical data behaves this way. Descriptors derived from the same atoms will always be entangled. Recognising the condition and knowing not to over-interpret individual coefficients is a genuine professional skill.

**Pharmacy analogy:** In a stability study where temperature and humidity rise and fall together in the chamber, both will correlate with degradation — and no statistical analysis can tell you which one actually caused it.

In [ ]:
# Block 5: Separate the input features (X) from the target variable (y)

feature_columns = ["molecular_weight", "logP", "num_h_donors",
                   "num_h_acceptors", "polar_surface_area",
                   "num_rotatable_bonds"]

X = df[feature_columns]

y = df["logS"]

print("X (input features) shape:", X.shape)
print("Feature names:", list(X.columns))
print()
print("y (target variable) shape:", y.shape)
print("Target name: logS")
print()
print("Note: Compound_Name and SMILES are deliberately excluded.")
print("They identify the compound but are not numerical properties.")

**Meaning of Block 5**

**Purpose:** Divide the table into the information the model may *look at*, and the value it must *predict*.

**What data comes IN:** `df` from Block 2.

**What happens inside — line by line:**

- `feature_columns = [...]` names the six descriptor columns explicitly, rather than dropping unwanted columns. Naming what you want is safer than naming what you do not want, because a column added to the file later cannot then slip into the model unnoticed.
- `df[feature_columns]` selects those six columns and stores them as `X`.
- `df["logS"]` selects the target column and stores it as `y`.

**What comes OUT:** `X` — a table of 1,128 rows × 6 columns. `y` — a single list of 1,128 values.

**Why the name and SMILES columns are excluded:** They identify each compound but are text, not measurable quantities. A model cannot calculate with the word "Testosterone". They remain in `df` so results can be traced back to real molecules, but they are never fed to the algorithm.

**The vocabulary of machine learning — learn these now, they recur in every practical:**

- **Feature** (also called an *input*, a *predictor*, an *independent variable*, or simply *X*): a piece of information given to the model so that it can make a prediction. Here there are six, all molecular descriptors.
- **Target** (also called the *output*, the *label*, the *dependent variable*, or simply *y*): the single value the model must learn to produce. Here it is logS.
- **Variable:** a named box in memory holding a value. `X` and `y` are both variables.
- **Sample** (or *instance*, or *observation*): one row — one compound with its six descriptors and its measured logS.

Capital `X` is written in upper case because it is a table with many columns; lower-case `y` because it is a single column. This convention appears in every machine learning textbook and every piece of code you will ever read.

**Why the separation must be exact:** If `logS` were accidentally left inside `X`, the model would receive the answer along with the question. It would score perfectly and would have learned nothing whatever. This mistake is called **data leakage**, and it is among the most common and most damaging errors in applied machine learning.

**Pharmacy analogy:** A physician examines symptoms, laboratory values and history — the features — to arrive at a diagnosis, the target. The diagnosis must not already be written on the front of the file.

In [ ]:
# Block 6: Split the data into a training set and a testing set

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape[0], "compounds")
print("Testing set :", X_test.shape[0], "compounds")
print()
print("The model will learn from the training set only.")
print("The testing set stays hidden until evaluation.")

**Meaning of Block 6**

**Purpose:** Reserve part of the data so the model can later be examined on compounds it has never seen.

**What data comes IN:** `X` and `y` from Block 5.

**What happens inside — line by line:**

- `train_test_split(...)` shuffles the 1,128 compounds and divides them into two groups, returning four objects at once.
- `test_size=0.20` sets the testing portion to 20 per cent. With 1,128 compounds this gives **902 for training and 226 for testing**.
- `random_state=42` fixes the shuffling so the split is identical every time the notebook runs. Without it, a different 226 compounds would be selected on each run and your results would shift slightly each time. The number 42 has no mathematical significance; any fixed number would serve equally. Setting it is what makes the practical **reproducible** — you, your classmate and your examiner will all obtain exactly the same figures.

**What comes OUT:**

- `X_train`, `y_train` — 902 compounds used for learning
- `X_test`, `y_test` — 226 compounds locked away for the final examination

**Why this step is non-negotiable:** A model can memorise. Tested on the same compounds it learned from, it could score beautifully by simple recall while being useless on anything new. That failure is called **overfitting**. Testing on held-out data is the only honest way to discover whether a model has genuinely learned the underlying relationship or has merely memorised the answers.

**Pharmacy analogy:** Students study from the textbook during the semester, but the final examination uses questions they have not seen. Anything else would measure memory rather than understanding.

In [ ]:
# Block 7: Create the Linear Regression model

model = LinearRegression()

print("Model created:", model)
print()
print("At this stage the model is empty.")
print("It has seen no data and cannot predict anything yet.")

**Meaning of Block 7**

**Purpose:** Create an empty Linear Regression model, ready to be trained.

**What data comes IN:** Nothing.

**What happens inside — line by line:**

- `LinearRegression()` creates a new, untrained model object and stores it in the variable `model`. Nothing has been learned yet; this simply sets up the mathematical framework.

**What Linear Regression actually is:** It assumes the target can be written as a straight-line combination of the features:

> logS = intercept + (c₁ × molecular_weight) + (c₂ × logP) + (c₃ × num_h_donors) + (c₄ × num_h_acceptors) + (c₅ × polar_surface_area) + (c₆ × num_rotatable_bonds)

The six values c₁ to c₆ are the **coefficients**, and the constant is the **intercept**. Together they are the model's seven unknowns. "Training" means finding the seven numbers that make this equation fit the training data as closely as possible.

Because there are six inputs rather than one, this is strictly **Multiple Linear Regression**. With a single input it would be Simple Linear Regression, drawing an ordinary straight line on a graph.

**Why begin with this method:** Linear Regression is the most transparent algorithm in all of machine learning. When training finishes you can read its equation and see precisely how each molecular property influences the prediction. The more powerful algorithms met later in this series — Random Forests, neural networks — usually predict more accurately but cannot be read in this way. Understanding a model you can see through is the right foundation for later trusting one you cannot.

**What comes OUT:** `model` — an untrained Linear Regression object.

**Pharmacy analogy:** Admitting a student to a course. Enrolled, but not yet taught anything.

In [ ]:
# Block 8: Train the model on the training data

model.fit(X_train, y_train)

print("Training complete.")
print()
print("The model has examined", X_train.shape[0], "compounds")
print("and calculated the equation that best links the six")
print("molecular descriptors to measured solubility.")

**Meaning of Block 8**

**Purpose:** This is the learning step — where the model actually acquires its knowledge.

**What data comes IN:** `X_train` and `y_train` from Block 6.

**What happens inside — line by line:**

- `model.fit(X_train, y_train)` is the single most important instruction in the notebook. The word *fit* means "find the equation that best fits this data".

Internally the algorithm searches for the seven numbers — six coefficients and one intercept — that make the total prediction error across all 902 training compounds as small as possible. The specific quantity minimised is the sum of the squared differences between predicted and measured logS, which is why the method is formally called **Ordinary Least Squares**.

Squaring the errors before adding them serves two purposes: it stops a positive error and a negative error cancelling one another out, and it penalises large mistakes far more heavily than small ones.

Notice that only the *training* compounds are supplied. The 226 test compounds are never mentioned, and the model remains entirely unaware that they exist.

**What comes OUT:** `model` — now trained, with its coefficients calculated and stored inside it.

**Pharmacy analogy:** The lectures and practical sessions of the semester. The student now holds the knowledge; the examination has not yet happened.

In [ ]:
# Block 9: Read the equation the model has learned

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_.round(5)
})

print("Learned coefficients:")
print(coefficients.to_string(index=False))
print()
print("Intercept =", round(model.intercept_, 5))
print()

print("The complete equation the model has learned:")
print()
equation = "logS = " + str(round(model.intercept_, 4))
for feature, coef in zip(X.columns, model.coef_):
    sign = "+" if coef >= 0 else "-"
    equation += "  " + sign + " " + str(abs(round(coef, 4))) + "*(" + feature + ")"
print(equation)

**Meaning of Block 9**

**Purpose:** Open up the trained model and read the equation it discovered. This is what makes Linear Regression genuinely interpretable.

**What data comes IN:** The trained `model` from Block 8.

**What happens inside — line by line:**

- `model.coef_` retrieves the six coefficients. The trailing underscore is a scikit-learn convention marking a value that was *learned from data* rather than set by you.
- `model.intercept_` retrieves the constant term.
- `pd.DataFrame({...})` pairs each coefficient with its feature name so the table reads easily.
- The `for` loop assembles and prints the full equation in ordinary algebraic form.

**What comes OUT:** `coefficients` — a readable table, plus the printed equation.

**How to interpret a coefficient:** Each coefficient answers one precise question — *if this descriptor increases by one unit and everything else stays the same, how much does predicted logS change?*

A **negative** coefficient means increasing that property makes the compound **less** soluble. A **positive** coefficient means it makes the compound **more** soluble.

**Reading the actual result:**

| Feature | Coefficient | Chemical reading |
|---|---|---|
| `logP` | **−0.93** | The dominant term. One extra logP unit lowers predicted logS by nearly a full unit — roughly a ninefold drop in solubility |
| `num_h_acceptors` | **+0.17** | Positive, as expected: acceptor atoms hydrogen-bond with water |
| `polar_surface_area` | **−0.016** | Negative — chemically counter-intuitive. See below |
| `num_h_donors` | **−0.087** | Negative — also counter-intuitive. See below |
| `molecular_weight` | **−0.004** | Small per unit, but molecular weight ranges over hundreds, so across a 300-unit span it contributes over a full log unit |
| `num_rotatable_bonds` | **−0.0002** | Essentially zero. This descriptor contributes nothing once the others are present |

**The logP result is the headline.** With a coefficient near −0.93 it is by far the strongest single influence, matching both the correlation seen in Block 4 and everything physical pharmaceutics teaches about lipophilicity opposing aqueous dissolution. The model discovered this from measurements alone, with no chemistry supplied to it.

**Now the awkward part — two coefficients have the wrong sign, and you should not ignore it.**

Hydrogen bond donors and polar surface area both came out slightly *negative*, implying they reduce solubility. Chemically that is nonsense: an –OH group helps a molecule dissolve in water, it does not hinder it.

This is **multicollinearity**, exactly as Block 4 predicted. Polar surface area and hydrogen bond acceptor count correlate at 0.90 — they are largely measuring the same oxygen and nitrogen atoms. When two features carry nearly the same information, the fitting procedure cannot determine which deserves the credit. It settles on whichever combination minimises total error, and a common outcome is a large positive coefficient on one paired with a compensating negative coefficient on the other. Together the pair predicts correctly; read separately, one of them lies.

**The professional conclusion:** in the presence of multicollinearity, individual coefficients cannot be interpreted as isolated chemical effects. The equation as a whole remains valid for prediction — the R² in Block 12 is real and honestly earned. But you must not quote "hydrogen bond donors reduce solubility" as a finding. It is an artefact of the mathematics, not a fact about chemistry.

Had the earlier practical used a tidy artificial dataset with independent descriptors, every coefficient would have carried its expected sign and this lesson would have been invisible. Real data is messier, and the mess is where the learning is.

**A further caution:** a coefficient describes an association within this dataset. It does not by itself establish a cause. Correlation and causation are different things, and no regression coefficient can distinguish them.

**Pharmacy analogy:** Identifying from a formulation study which excipient most influences dissolution — while remembering that if two excipients were always varied together, the study cannot separate their individual contributions no matter how the numbers are analysed.

In [ ]:
# Block 10: Use the trained model to predict the test compounds

y_pred = model.predict(X_test)

print("Predictions generated for", len(y_pred), "unseen test compounds.")
print()
print("First five predicted logS values:")
print(np.round(y_pred[:5], 3))

**Meaning of Block 10**

**Purpose:** Ask the trained model to predict logS for the 226 compounds it has never seen.

**What data comes IN:** The trained `model`, and `X_test` from Block 6.

**What happens inside — line by line:**

- `model.predict(X_test)` feeds each test compound's six descriptors into the learned equation and calculates a logS for it.
- Only `X_test` is supplied. The true answers in `y_test` are **not** given to the model. It is predicting blind, exactly as it would for a compound that had never been measured.

**What comes OUT:** `y_pred` — 226 predicted logS values, in the same order as the test compounds.

**Why this is the moment that matters:** Everything so far has been preparation. This is the model doing the job it was built for — estimating a property it was never told, for molecules it has never encountered.

**Pharmacy analogy:** A newly synthesised compound arrives with its physicochemical properties calculated but no solubility data. The model estimates its solubility before any dissolution experiment is set up.

In [ ]:
# Block 11: Compare predictions with the true values, compound by compound

results = pd.DataFrame({
    "Compound": df.loc[X_test.index, "Compound_Name"].values,
    "Actual logS": y_test.values.round(2),
    "Predicted logS": y_pred.round(2)
})

results["Error"] = (results["Predicted logS"] - results["Actual logS"]).round(2)

print("First 10 test compounds:")
print(results.head(10).to_string(index=False))
print()
print("Largest over-prediction :", results["Error"].max())
print("Largest under-prediction:", results["Error"].min())
print()
print("Five worst predictions:")
worst = results.reindex(results["Error"].abs().sort_values(ascending=False).index)
print(worst.head(5).to_string(index=False))

**Meaning of Block 11**

**Purpose:** Place predictions beside true values, named compound by named compound, so accuracy can be inspected directly.

**What data comes IN:** `y_test` from Block 6, `y_pred` from Block 10, and the compound names from `df`.

**What happens inside — line by line:**

- `df.loc[X_test.index, "Compound_Name"]` looks up the name of each test compound. Because `X_test` retained its original row numbers when the data was split, those numbers can be used to recover the names. This is why keeping identifier columns in `df` was worthwhile.
- The `pd.DataFrame({...})` line builds a table of name, actual and predicted values side by side.
- `results["Error"] = ...` adds a fourth column: predicted minus actual. A **positive** error means the model predicted the compound to be *more* soluble than it truly is; a **negative** error means *less* soluble.
- The final three lines re-sort the table by the size of the error, ignoring sign, so the five worst predictions can be inspected.

**What comes OUT:** `results` — a named comparison table.

**How to read it:** For a well-performing model, most errors should be small and should fall on both sides of zero. If nearly every error carried the same sign, the model would be systematically biased — consistently over- or under-predicting — which is a more serious defect than random scatter.

Note that a logS error of 1.0 is not small: because the scale is logarithmic, being wrong by one log unit means being wrong about the actual solubility by a factor of ten.

**Why looking at the worst predictions is worth your time:** The failures are more informative than the successes. Examine the five worst compounds and ask what they have in common. Are they unusually large? Highly charged? Do they contain metal atoms, or unusual ring systems? Real solubility depends on crystal lattice energy, polymorphic form and ionisation state — none of which appear among our six descriptors. The compounds where those hidden factors dominate are precisely the ones this model cannot get right, and identifying them tells you where the model's boundary lies.

**Pharmacy analogy:** Comparing a tablet's measured dissolution profile against the profile predicted during formulation development — and paying closest attention to the batches that disagreed.

In [ ]:
# Block 12: Evaluate the model with numerical metrics

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MODEL PERFORMANCE ON THE 226 UNSEEN TEST COMPOUNDS")
print("=" * 58)
print("Mean Absolute Error (MAE)      :", round(mae, 3), "log units")
print("Mean Squared Error (MSE)       :", round(mse, 3))
print("Root Mean Squared Error (RMSE) :", round(rmse, 3), "log units")
print("R-squared Score (R2)           :", round(r2, 3))
print("=" * 58)
print()
print("Interpretation:")
print("On average the prediction is off by about", round(mae, 2), "log units,")
print("which is a factor of about", round(10 ** mae, 1), "in actual solubility.")
print("The model explains about", round(r2 * 100, 1), "percent of the")
print("variation in solubility across the test compounds.")

**Meaning of Block 12**

**Purpose:** Replace visual impression with objective numbers.

**What data comes IN:** `y_test` and `y_pred`.

**What happens inside — the four metrics explained:**

**Mean Absolute Error (MAE) — about 0.84 log units.** Take every error, ignore whether it is positive or negative, and average them. This is the most intuitive metric: on average, predictions miss by about 0.84 log units, which corresponds to being wrong about the true solubility by roughly sevenfold. It is expressed in the same units as logS.

**Mean Squared Error (MSE) — about 1.22.** Square each error before averaging. Squaring makes large mistakes count disproportionately, so MSE is sensitive to occasional bad predictions. Its units are logS *squared*, which has no physical meaning, so it is rarely quoted alone.

**Root Mean Squared Error (RMSE) — about 1.11 log units.** The square root of MSE, which returns the value to the original logS units so it can be read directly. RMSE is always greater than or equal to MAE, and the gap between them shows how much a few large errors are contributing. Here 1.11 against 0.84 is a noticeable gap, telling you that a minority of compounds are predicted considerably worse than the typical one — which is exactly what the worst-prediction list in Block 11 revealed.

**R-squared (R²) — about 0.742.** This answers: what fraction of the variation in logS does the model successfully explain?

- R² = 1.0 would be perfect prediction
- R² = 0.0 would mean the model performs no better than always guessing the average
- R² can even be negative, meaning it performs *worse* than guessing the average

An R² of 0.742 means about 74 per cent of the variation is explained.

**How good is 0.742, honestly? This is a genuinely good result, and here is why.**

It may look modest beside the near-perfect scores that appear in textbook examples, but those examples almost always use artificial data. On the Delaney dataset, a linear model using simple descriptors of this kind typically achieves R² in the region of 0.7 to 0.8, and Delaney's own published ESOL method sits in the same territory. Your model is performing at the level of published work in the field.

The remaining 26 per cent is not a failure of your code. It is the part of solubility that six descriptors cannot reach. Real aqueous solubility depends on crystal lattice energy — how tightly molecules pack in the solid state — on polymorphic form, on ionisation at the relevant pH, and on the entropy of dissolution. None of these can be calculated from molecular weight and logP. A model can only ever explain what its inputs contain.

**The most important habit to take from this block:** always ask what a reported R² was measured on. A score of 0.95 on data generated by a formula proves nothing. A score of 0.74 on 1,128 real laboratory measurements is a genuine, usable result. The larger number is the weaker claim.

**Pharmacy analogy:** Validating an analytical method by calculating accuracy, precision and linearity before release for routine use. The numbers, not the analyst's impression, decide whether the method is fit for purpose — and a method honest about its limits is safer than one that appears flawless.

In [ ]:
# Block 13: Confirm the result with cross-validation

cv_scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')

print("5-FOLD CROSS-VALIDATION")
print("=" * 58)
for i, score in enumerate(cv_scores, start=1):
    print("Fold", i, "R-squared :", round(score, 3))
print("=" * 58)
print("Mean R-squared     :", round(cv_scores.mean(), 3))
print("Standard deviation :", round(cv_scores.std(), 3))
print()
print("A small standard deviation means the result is stable")
print("and was not produced by one lucky split.")

**Meaning of Block 13**

**Purpose:** Test whether the result in Block 12 was genuine, or simply a fortunate choice of test compounds.

**What data comes IN:** The full `X` and `y` from Block 5 — not the split versions.

**What happens inside — line by line:**

- `cross_val_score(...)` performs the entire train-and-test procedure five separate times.
- `cv=5` means the 1,128 compounds are divided into five roughly equal groups, called **folds**. The model trains on four folds and is tested on the fifth; this repeats five times so that every compound serves in a test set exactly once.
- `scoring='r2'` requests the R² of each round.
- A brand-new `LinearRegression()` is created inside the function each time, so no knowledge carries over between rounds.

**What comes OUT:** `cv_scores` — five R² values, one per fold.

**Why this step matters:** Block 6 made a single split with `random_state=42`. Any one split can be unusually easy or unusually hard, making a model look better or worse than it truly is. Cross-validation removes that luck by averaging across five different splits.

**How to read the result:** The five scores fall between roughly 0.73 and 0.82, giving a mean near 0.77 with a standard deviation of about 0.03. The spread is narrow, and the mean sits close to the 0.742 obtained from the single split. The conclusion is that the earlier figure was a fair representation and not an accident.

Had the folds instead ranged from 0.3 to 0.9, that wide scatter would be a warning: the model would be unstable and any single reported score untrustworthy.

**A subtle point worth noticing:** the cross-validated mean of 0.77 is slightly *higher* than the single-split 0.742. This simply means the particular 226 compounds chosen by `random_state=42` happened to be a slightly harder set than average. Reporting only the more flattering of the two numbers would be a small dishonesty. The professional practice is to report both, and to quote the cross-validated figure as the model's true expected performance.

**Pharmacy analogy:** Analytical method validation is never based on a single injection. The determination is repeated on different days, by different analysts, on different instruments — and only if the results agree is the method declared reliable.

In [ ]:
# Block 14: Plot actual versus predicted solubility

plt.figure(figsize=(7, 6))

plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='k', linewidths=0.4)

lo = min(y_test.min(), y_pred.min()) - 0.5
hi = max(y_test.max(), y_pred.max()) + 0.5
plt.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect prediction')

plt.xlabel("Measured logS")
plt.ylabel("Predicted logS")
plt.title("Measured vs Predicted Aqueous Solubility (226 test compounds)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Meaning of Block 14**

**Purpose:** See the model's performance at a glance, in a single figure.

**What data comes IN:** `y_test` and `y_pred`.

**What happens inside — line by line:**

- `plt.scatter(y_test, y_pred, ...)` plots one dot per test compound: its measured logS along the horizontal axis, its predicted logS up the vertical axis.
- `alpha=0.5` makes the dots semi-transparent so that overlapping points remain distinguishable — important with 226 compounds on one figure.
- `plt.plot([lo, hi], [lo, hi], 'r--')` draws a red dashed diagonal. This is the line of **perfect prediction**: any compound predicted exactly right would sit precisely on it.
- `plt.legend()`, `plt.xlabel()`, `plt.ylabel()` and `plt.title()` label the figure. An unlabelled graph is not acceptable in a scientific report.

**What comes OUT:** The scatter plot.

**How to read it:**

- Points **on** the red line are perfect predictions
- Points **above** the line were predicted more soluble than they truly are
- Points **below** the line were predicted less soluble than they truly are
- The tighter the cloud hugs the line, the better the model

In this result the points follow the diagonal clearly across the whole thirteen-log-unit range, with a visible band of scatter around it. That band is the ±0.84 log unit typical error made visible. Compare it mentally with a textbook figure showing points sitting almost exactly on the line: this is what a real, honest model looks like, and it is still highly useful.

**What a bad plot would look like — worth recognising:** A curved, banana-shaped cloud would indicate that the true relationship is not linear and that a straight-line model is the wrong tool. A cloud fanning out wider at one end would indicate that the model is far less reliable in that part of the solubility range.

**Pharmacy analogy:** Plotting a calibration curve during analytical method development. A straight line of tightly clustered points signals a sound method; scatter or curvature signals a problem that no amount of arithmetic will hide.

In [ ]:
# Block 15: Plot the residuals to check for hidden bias

residuals = y_test.values - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.5, edgecolors='k', linewidths=0.4)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel("Predicted logS")
plt.ylabel("Residual (Measured - Predicted)")
plt.title("Residual Plot")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Mean residual     :", round(residuals.mean(), 4))
print("Residual std dev  :", round(residuals.std(), 4))
print()
print("A mean close to zero means the model is not")
print("systematically over- or under-predicting.")

**Meaning of Block 15**

**Purpose:** Look for systematic error that the previous plot could conceal.

**What data comes IN:** `y_test` and `y_pred`.

**What happens inside — line by line:**

- `residuals = y_test.values - y_pred` computes, for each compound, how far the truth sits from the prediction. A **residual** is simply a leftover error.
- `plt.scatter(y_pred, residuals, ...)` plots each residual against the predicted value.
- `plt.axhline(y=0, ...)` draws a red horizontal line at zero — the position of a perfect prediction.

**What comes OUT:** The residual plot, plus the mean and spread of the residuals.

**Why a separate plot is needed:** The measured-versus-predicted plot in Block 14 is dominated by the overall trend, and a modest systematic bias can hide inside it. A residual plot removes the trend entirely and magnifies whatever is left, making faults far easier to see.

**How to read it — you want to see nothing:** A healthy residual plot is a shapeless, random cloud spread evenly above and below the red line, with roughly constant thickness from left to right. Structure is the warning sign:

- A **curved or U-shaped** band means the true relationship is not linear, and a linear model is systematically wrong at the extremes.
- A **funnel or cone shape**, widening to one side, means the errors grow larger in that part of the range. This is called **heteroscedasticity**, and it means the model's reliability is not uniform.
- A **tilt**, with residuals drifting upward or downward across the plot, means the model consistently over-predicts in one region and under-predicts in another.

Here the mean residual is essentially zero, confirming there is no overall bias in either direction. Look carefully at the shape of the cloud as well, particularly at the extreme left where the least soluble compounds sit — with real data, the very insoluble compounds are often the hardest to predict, and any widening there tells you the model should be trusted less in that region.

**Why this matters practically:** If you were screening candidate molecules and the model were systematically over-optimistic about poorly soluble compounds, you would advance compounds that later fail in formulation. A residual plot is how you find that out before it costs anything.

**Pharmacy analogy:** In analytical method validation, plotting the residuals of a calibration curve reveals curvature at high concentration that the correlation coefficient alone would happily disguise.

In [ ]:
# Block 16: Save the trained model for future use

joblib.dump(model, "solubility_linear_regression_model.pkl")

feature_order = list(X.columns)
joblib.dump(feature_order, "solubility_feature_order.pkl")

print("Saved two files:")
print("  solubility_linear_regression_model.pkl  - the trained model")
print("  solubility_feature_order.pkl            - the required column order")
print()
print("Required feature order:", feature_order)
print()
print("These can be reloaded later with joblib.load()")
print("to make predictions without retraining.")

# To download them to your own computer, remove the # from the next three lines
# from google.colab import files
# files.download("solubility_linear_regression_model.pkl")
# files.download("solubility_feature_order.pkl")

**Meaning of Block 16**

**Purpose:** Store the trained model on disk so it can be reused without repeating the training.

**What data comes IN:** The trained `model`, and the column names from `X`.

**What happens inside — line by line:**

- `joblib.dump(model, "....pkl")` writes the entire trained model — its coefficients, its intercept, its internal settings — into a single file. The `.pkl` extension stands for *pickle*, Python's term for converting a live object into a storable file.
- The second `joblib.dump(...)` saves the list of feature names **in their exact order**. This matters more than it appears: the model identifies features purely by position, not by name. Supplying the six descriptors in a different order would produce a confident but entirely wrong prediction, with no error message to warn you.
- The three commented lines at the end will download the files to your computer once the `#` symbols are removed.

**What comes OUT:** Two saved files.

**Why saving matters:** Training this model takes under a second, so saving it may seem unnecessary here. But the habit is essential. Deep learning models later in this series can take many minutes or hours to train, and retraining for every single prediction would be absurd. Saving also guarantees that the exact model you validated is the exact model you later deploy — an important point in any regulated environment, where you must be able to demonstrate that the model behind a decision is the one that was tested.

**Pharmacy analogy:** Recording a validated analytical method in an SOP. The method is developed once, documented, and thereafter followed exactly by every analyst.

In [ ]:
# ============================================================
# Block 17: PREDICT THE SOLUBILITY OF A COMPOUND OF YOUR CHOICE
# ============================================================
#
#  *** DEMONSTRATION VALUES BELOW ***
#
#  The six values below belong to TESTOSTERONE, included only
#  as a worked example. Testosterone was part of the held-out
#  test set, so the model has never learned from it - which
#  means we can honestly compare the prediction against its
#  real measured value.
#
#  REPLACE THEM WITH YOUR OWN COMPOUND'S VALUES before drawing
#  any conclusion. Every line marked  <-- EDIT  is yours to change.
#
# ============================================================

compound_name = "Testosterone"      # <-- EDIT: name of your compound

new_compound = pd.DataFrame({
    "molecular_weight":    [288.43],   # <-- EDIT: g/mol
    "logP":                [3.880],    # <-- EDIT: octanol/water partition coefficient
    "num_h_donors":        [1],        # <-- EDIT: count of -OH and -NH groups
    "num_h_acceptors":     [2],        # <-- EDIT: count of O and N atoms
    "polar_surface_area":  [37.30],    # <-- EDIT: TPSA in square Angstroms
    "num_rotatable_bonds": [0]         # <-- EDIT: count of freely rotating single bonds
})

# ---- Confirmation echo: check these are the values you intended ----
print("YOU ENTERED:")
print("-" * 58)
print("Compound name       :", compound_name)
for column in new_compound.columns:
    print(f"{column:<22}: {new_compound[column].values[0]}")
print("-" * 58)
print()

# ---- Warn if the demonstration values are still in place ----
if compound_name == "Testosterone":
    print("!" * 58)
    print("NOTE: You are still using the demonstration compound.")
    print("Edit the values above to predict your own compound.")
    print("!" * 58)
    print()

# ---- Make the prediction ----
predicted_logS = model.predict(new_compound)[0]

print("PREDICTED RESULT")
print("=" * 58)
print("Predicted logS :", round(predicted_logS, 3))
print("Molar solubility:", f"{10 ** predicted_logS:.2e}", "mol/L")
print("Typical error of this model: plus or minus",
      round(mae, 2), "log units")
print("=" * 58)

# ---- If the compound is in our dataset, show the measured value ----
match = df[df["Compound_Name"].str.lower() == compound_name.lower()]
if len(match) > 0:
    measured = match["logS"].values[0]
    print()
    print("This compound appears in the dataset, so we can check:")
    print("  Measured logS  :", measured)
    print("  Predicted logS :", round(predicted_logS, 3))
    print("  Error          :", round(predicted_logS - measured, 3), "log units")

**Meaning of Block 17**

**Purpose:** Apply the finished model to a compound of your own choosing — the practical payoff of everything built so far.

**What data comes IN:** The trained `model`, plus six descriptor values that you supply.

**What happens inside — line by line:**

- `compound_name` and `new_compound` hold your input. Every editable line carries an `<-- EDIT` marker.
- `pd.DataFrame({...})` wraps the six values into a one-row table. The model expects a table, not six loose numbers, and the column names must match those used in training **exactly**.
- The confirmation echo prints your values straight back at you. This is deliberate: a mistyped digit is otherwise invisible, and the model will never object — it will simply return a confident, wrong answer.
- The `if` statement warns you while the demonstration values are still in place, so a demonstration result is never mistaken for your own.
- `model.predict(new_compound)[0]` substitutes your six numbers into the learned equation. The `[0]` extracts the single value from the list returned.
- `10 ** predicted_logS` converts logS back into ordinary molar solubility, since logS is a base-10 logarithm.
- The final block searches the dataset for your compound by name. If it is present, the measured value is displayed alongside the prediction so the error can be seen directly.

**What comes OUT:** A predicted logS, the equivalent molar solubility, and — for a compound in the dataset — the measured value for comparison.

**About the demonstration compound:** Testosterone is the principal male sex hormone and a widely used therapeutic agent. It is a good example precisely because it is a genuine drug molecule, it appeared in the **held-out test set**, and its solubility was therefore never shown to the model during training. The prediction below is a real blind prediction, not a recollection.

**The result, and how to read it honestly:**

The model predicts a logS of about **−4.80**. The measured value in the dataset is **−4.02**. The prediction is therefore too low by roughly 0.78 log units, meaning the model expects testosterone to be about six times less soluble than it actually is.

Is that a good result or a bad one? It is a **typical** one, and typical is the honest answer. The model's mean absolute error across all 226 test compounds is 0.84 log units, so testosterone's error of 0.78 sits right at the average. The model behaved exactly as its validation said it would.

**What this means for how you use such a model:**

- Treat the prediction as an **estimate with a stated uncertainty**, never as a measurement. Quoting "logS = −4.80" alone is misleading; "logS ≈ −4.8, typical error ±0.8 log units" is honest.
- A model of this accuracy is genuinely useful for **ranking and triage** — deciding which of two hundred candidate molecules are worth measuring first. It is not accurate enough to set a formulation specification.
- The remaining error is not carelessness. Aqueous solubility also depends on crystal lattice energy, polymorphic form and ionisation state, and none of those can be calculated from molecular weight and logP. Six numbers cannot capture everything a molecule does in water.
- **Every computational prediction requires experimental confirmation** before any formulation decision rests on it.

**Try this yourself:** replace the values above with another compound from the dataset — phenytoin, warfarin, diazepam and caffeine are all present — and see how the error varies. Then try a compound that is *not* in the dataset at all and notice that you have no way of checking the answer. That is the ordinary situation in real drug discovery, and it is why validation on held-out data matters so much.

**Pharmacy analogy:** A validated dissolution method reports a result with a known confidence interval. An analyst who quotes the number and omits the interval has not reported the result — they have reported half of it.

---

## Summary of this practical

| Item | Detail |
|---|---|
| Practical type | **Regression** — the model predicts a continuous number, not a category |
| Approach | **Machine Learning (ML)** — not deep learning; no neural network is used |
| Learning type | **Supervised** — every training compound came with its correct measured logS |
| Algorithm | Multiple Linear Regression (Ordinary Least Squares) |
| Input features (X) | 6 molecular descriptors |
| Target variable (y) | logS, the base-10 logarithm of molar aqueous solubility |
| Dataset | Delaney (ESOL), 1,128 real compounds with measured solubility |
| Split | 902 training / 226 testing (80:20) |
| Performance | R² ≈ 0.74, MAE ≈ 0.84 log units, RMSE ≈ 1.11 log units |
| Cross-validated | 5-fold mean R² ≈ 0.77, standard deviation ≈ 0.03 |

**What you have learned to do**

You have taken a table of real laboratory measurements, examined it, checked how its variables relate to one another, separated features from target, held part of it back, trained a model, read the equation it discovered, recognised where that equation could mislead you, measured its accuracy four different ways, confirmed the result was not luck, diagnosed it with two plots, saved it, and applied it to a drug molecule it had never seen — comparing the answer honestly against the truth.

That sequence is the standard machine learning workflow. Every remaining practical in this series follows the same shape, with different data and more powerful algorithms.

**The three ideas most worth carrying forward**

1. **A model can only learn what its inputs contain.** Twenty-six per cent of solubility remained unexplained because crystal packing and ionisation are not in our six descriptors.
2. **When features overlap, coefficients stop being chemistry.** Multicollinearity kept the equation predictive while making two of its terms uninterpretable.
3. **Always ask what a score was measured on.** R² 0.74 on real experimental data is a stronger claim than R² 0.95 on data made by a formula.